# Milestone 2 — Frontier Text-to-SQL baseline

## Checkpoint 1: One-question vertical slice

This notebook establishes the simplest credible frontier-model baseline.

GPT-5.2 receives only:

- the natural-language question;
- table and column names from the SQLite database;
- a short instruction to return one SQLite query.

It does **not** receive:

- reference SQL or reference results;
- example question/SQL pairs;
- sample database values;
- domain explanations or a glossary;
- retrieval, planning, retries, or SQL repair.

The generated SQL is executed exactly as returned and compared with the audited
reference answer using the Milestone 1 evaluator.

In [19]:
from pathlib import Path
import json
import os
import sqlite3

from azure.identity import AzureCliCredential, get_bearer_token_provider
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI

from domain_adapt.evaluation import compare_sql_results
from domain_adapt.finch import make_reference_compatible


load_dotenv(find_dotenv(usecwd=True))

dataset_path = Path("../data/finch_dataset.json").resolve()
db_path = Path("../data/financial.sqlite").resolve()
manifest_path = Path("../configs/eval_manifest_v1.json").resolve()

deployment_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]
tenant_id = os.environ["AZURE_TENANT_ID"]

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
if not endpoint.endswith("/openai/v1"):
    endpoint = f"{endpoint}/openai/v1"

credential = AzureCliCredential(tenant_id=tenant_id)

token_provider = get_bearer_token_provider(
    credential,
    "https://ai.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint}/",
    api_key=token_provider,
)

print("Deployment:", deployment_name)
print("Database:", db_path.name)

Deployment: gpt-5.2
Database: financial.sqlite


In [7]:
records = json.loads(dataset_path.read_text(encoding="utf-8"))
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

working_ids = set(
    manifest["selection"]["working_dev"]["ids"]
)
frozen_ids = set(
    manifest["selection"]["frozen_eval"]["ids"]
)

question_id = 16

assert question_id in working_ids
assert question_id not in frozen_ids

matches = [
    row
    for row in records
    if row["db_name"] == "bird"
    and row["db_id"] == "financial"
    and row["question_id"] == question_id
]

assert len(matches) == 1
example = matches[0]

print("Question ID:", example["question_id"])
print("Difficulty:", example["difficulty"])
print("Question:", example["question"])

Question ID: 16
Difficulty: easy
Question: The transaction of 840 USD happened in 1998/10/14, when was this account opened?


In [8]:
conn = sqlite3.connect(
    f"file:{db_path.as_posix()}?mode=ro",
    uri=True,
)

table_names = [
    row[0]
    for row in conn.execute(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
          AND name NOT LIKE 'sqlite_%'
        ORDER BY name
        """
    )
]

schema_lines = []

for table_name in table_names:
    columns = conn.execute(
        f'PRAGMA table_info("{table_name}")'
    ).fetchall()

    column_definitions = ", ".join(
        f"{column[1]} {column[2]}"
        for column in columns
    )

    schema_lines.append(
        f"{table_name}({column_definitions})"
    )

schema_context = "\n".join(schema_lines)

print(schema_context)

account(account_id INTEGER, district_id INTEGER, frequency TEXT, date DATE)
card(card_id INTEGER, disp_id INTEGER, type TEXT, issued DATE)
client(client_id INTEGER, gender TEXT, birth_date DATE, district_id INTEGER)
disp(disp_id INTEGER, client_id INTEGER, account_id INTEGER, type TEXT)
district(district_id INTEGER, A2 TEXT, A3 TEXT, A4 TEXT, A5 TEXT, A6 TEXT, A7 TEXT, A8 INTEGER, A9 INTEGER, A10 REAL, A11 INTEGER, A12 REAL, A13 REAL, A14 INTEGER, A15 INTEGER, A16 INTEGER)
loan(loan_id INTEGER, account_id INTEGER, date DATE, amount INTEGER, duration INTEGER, payments REAL, status TEXT)
order(order_id INTEGER, account_id INTEGER, bank_to TEXT, account_to INTEGER, amount REAL, k_symbol TEXT)
trans(trans_id INTEGER, account_id INTEGER, date DATE, type TEXT, operation TEXT, amount INTEGER, balance INTEGER, k_symbol TEXT, bank TEXT, account INTEGER)


In [9]:
BASELINE_INSTRUCTIONS = """
You translate natural-language questions into SQLite SQL.

Return exactly one executable SQLite query.
Use only the tables and columns in the supplied schema.
Do not include Markdown, code fences, comments, or explanations.
Do not invent tables or columns.
""".strip()

model_input = f"""
Database schema:

{schema_context}

Question:

{example["question"]}
""".strip()

response = client.responses.create(
    model=deployment_name,
    instructions=BASELINE_INSTRUCTIONS,
    input=model_input,
    max_output_tokens=2000,
    store=False,
)

generated_sql = response.output_text.strip()

print("Response ID:", response.id)
print("Response status:", response.status)
print("\nRaw model output:")
print(generated_sql)
print("\nUsage:")
print(response.usage)

Response ID: resp_04e3efacdbdc8608016ab778af9f788195b458b50c4e455d92
Response status: completed

Raw model output:
SELECT a.date
FROM trans t
JOIN account a ON a.account_id = t.account_id
WHERE t.amount = 840
  AND t.date = '1998-10-14'
LIMIT 1;

Usage:
ResponseUsage(input_tokens=296, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=48, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=344)


In [10]:
compatible_reference_sql, applied_reference_rules = (
    make_reference_compatible(example["SQL"])
)

comparison = compare_sql_results(
    conn=conn,
    reference_sql=compatible_reference_sql,
    candidate_sql=generated_sql,
)

print("Question ID:", question_id)
print("Reference-only compatibility rules:", applied_reference_rules)
print("Passed:", comparison.equivalent)
print("Reason:", comparison.reason)
print("Expected row count:", comparison.expected_row_count)
print("Candidate row count:", comparison.candidate_row_count)

Question ID: 16
Reference-only compatibility rules: ['case_insensitive_text_equality']
Passed: True
Reason: unordered results match
Expected row count: 1
Candidate row count: 1


## Checkpoint 2: Measure the complete working set

The prompt contract is now fixed. We run one independent, single-attempt request
for each of the 12 visible `working_dev` questions.

Each generated query is evaluated exactly as returned. Failures are preserved
because they reveal what the minimal baseline does not understand.

This is development-set measurement, not the final frozen evaluation.

In [11]:
from collections import Counter
from datetime import datetime, timezone
from time import perf_counter


working_examples = sorted(
    [
        row
        for row in records
        if row["db_name"] == "bird"
        and row["db_id"] == "financial"
        and row["question_id"] in working_ids
    ],
    key=lambda row: row["question_id"],
)

assert len(working_examples) == 12
assert {row["question_id"] for row in working_examples} == working_ids

working_results = []

for index, working_example in enumerate(working_examples, start=1):
    model_input = f"""
Database schema:

{schema_context}

Question:

{working_example["question"]}
""".strip()

    started = perf_counter()

    try:
        response = client.responses.create(
            model=deployment_name,
            instructions=BASELINE_INSTRUCTIONS,
            input=model_input,
            max_output_tokens=2000,
            store=False,
        )

        latency_seconds = perf_counter() - started
        generated_sql = response.output_text.strip()

        compatible_reference_sql, applied_rules = (
            make_reference_compatible(working_example["SQL"])
        )

        comparison = compare_sql_results(
            conn=conn,
            reference_sql=compatible_reference_sql,
            candidate_sql=generated_sql,
        )

        usage = (
            response.usage.model_dump()
            if response.usage is not None
            else {}
        )

        result = {
            "question_id": working_example["question_id"],
            "difficulty": working_example["difficulty"],
            "question": working_example["question"],
            "generated_sql": generated_sql,
            "passed": comparison.equivalent,
            "reason": comparison.reason,
            "expected_row_count": comparison.expected_row_count,
            "candidate_row_count": comparison.candidate_row_count,
            "reference_compatibility_rules": applied_rules,
            "response_id": response.id,
            "response_status": response.status,
            "response_model": response.model,
            "latency_seconds": round(latency_seconds, 3),
            "usage": usage,
            "api_error": None,
        }

    except Exception as error:
        latency_seconds = perf_counter() - started

        result = {
            "question_id": working_example["question_id"],
            "difficulty": working_example["difficulty"],
            "question": working_example["question"],
            "generated_sql": None,
            "passed": False,
            "reason": "model request failed",
            "expected_row_count": None,
            "candidate_row_count": None,
            "reference_compatibility_rules": None,
            "response_id": None,
            "response_status": None,
            "response_model": None,
            "latency_seconds": round(latency_seconds, 3),
            "usage": {},
            "api_error": f"{type(error).__name__}: {error}",
        }

    working_results.append(result)

    outcome = "PASS" if result["passed"] else "FAIL"
    print(
        f"[{index:02d}/{len(working_examples)}] "
        f"ID {result['question_id']:>3} "
        f"({result['difficulty']:<6}) "
        f"{outcome}: {result['reason']}"
    )

[01/12] ID   5 (medium) PASS: unordered results match
[02/12] ID  11 (medium) FAIL: ordered results differ
[03/12] ID  14 (easy  ) FAIL: unordered results differ
[04/12] ID  16 (easy  ) PASS: unordered results match
[05/12] ID  45 (easy  ) FAIL: ordered results differ
[06/12] ID  49 (medium) FAIL: unordered results differ
[07/12] ID  61 (hard  ) FAIL: unordered results differ
[08/12] ID  69 (easy  ) PASS: unordered results match
[09/12] ID  81 (hard  ) FAIL: unordered results differ
[10/12] ID  89 (easy  ) PASS: unordered results match
[11/12] ID  99 (easy  ) FAIL: unordered results differ
[12/12] ID 105 (medium) FAIL: column count differs: 3 != 2


In [12]:
passed_count = sum(result["passed"] for result in working_results)
total_count = len(working_results)

difficulty_summary = {}

for difficulty in ("easy", "medium", "hard"):
    difficulty_results = [
        result
        for result in working_results
        if result["difficulty"] == difficulty
    ]

    difficulty_summary[difficulty] = {
        "passed": sum(result["passed"] for result in difficulty_results),
        "total": len(difficulty_results),
    }

failure_reasons = Counter(
    result["reason"]
    for result in working_results
    if not result["passed"]
)

total_usage = {
    "input_tokens": sum(
        result["usage"].get("input_tokens", 0)
        for result in working_results
    ),
    "output_tokens": sum(
        result["usage"].get("output_tokens", 0)
        for result in working_results
    ),
    "reasoning_tokens": sum(
        result["usage"]
        .get("output_tokens_details", {})
        .get("reasoning_tokens", 0)
        for result in working_results
    ),
    "total_tokens": sum(
        result["usage"].get("total_tokens", 0)
        for result in working_results
    ),
}

print(f"Overall: {passed_count}/{total_count}")
print(f"Accuracy: {passed_count / total_count:.1%}")
print("By difficulty:", difficulty_summary)
print("Failure reasons:", dict(failure_reasons))
print("Token usage:", total_usage)

Overall: 4/12
Accuracy: 33.3%
By difficulty: {'easy': {'passed': 3, 'total': 6}, 'medium': {'passed': 1, 'total': 4}, 'hard': {'passed': 0, 'total': 2}}
Failure reasons: {'ordered results differ': 2, 'unordered results differ': 5, 'column count differs: 3 != 2': 1}
Token usage: {'input_tokens': 3572, 'output_tokens': 938, 'reasoning_tokens': 0, 'total_tokens': 4510}


In [13]:
run = {
    "run_version": 1,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "benchmark": manifest["benchmark"],
    "split": "working_dev",
    "deployment_name": deployment_name,
    "declared_underlying_model": "gpt-5.2",
    "declared_model_version": "2025-12-11",
    "prompt_contract": {
        "instructions": BASELINE_INSTRUCTIONS,
        "schema_context": schema_context,
        "max_output_tokens": 2000,
        "few_shot_examples": 0,
        "candidate_sql_repair": False,
    },
    "summary": {
        "passed": passed_count,
        "total": total_count,
        "accuracy": passed_count / total_count,
        "by_difficulty": difficulty_summary,
        "failure_reasons": dict(failure_reasons),
        "usage": total_usage,
    },
    "results": working_results,
}

run_path = Path(
    "../data/runs/gpt_5_2_minimal_working_dev_v1.json"
).resolve()

run_path.parent.mkdir(parents=True, exist_ok=True)
run_path.write_text(
    json.dumps(run, indent=2),
    encoding="utf-8",
)

print("Saved:", run_path)

Saved: C:\Users\shchitt\Downloads\Projects\domain-adapt\data\runs\gpt_5_2_minimal_working_dev_v1.json


In [14]:
failures = [
    result
    for result in working_results
    if not result["passed"]
]

print("Failure count:", len(failures))

for failure in failures:
    print("=" * 80)
    print("Question ID:", failure["question_id"])
    print("Difficulty:", failure["difficulty"])
    print("Question:", failure["question"])
    print("Generated SQL:", failure["generated_sql"])
    print("Failure reason:", failure["reason"])
    print("API error:", failure["api_error"])

Failure count: 8
Question ID: 11
Difficulty: medium
Question: Among the accounts who have loan validity more than 12 months, list out the accounts that have the highest approved amount and have account opening date in 1993.
Generated SQL: SELECT a.account_id
FROM account AS a
JOIN loan AS l ON l.account_id = a.account_id
WHERE l.duration > 12
  AND l.status = 'A'
  AND strftime('%Y', a.date) = '1993'
  AND l.amount = (
    SELECT MAX(l2.amount)
    FROM loan AS l2
    JOIN account AS a2 ON a2.account_id = l2.account_id
    WHERE l2.duration > 12
      AND l2.status = 'A'
      AND strftime('%Y', a2.date) = '1993'
  );
Failure reason: ordered results differ
API error: None
Question ID: 14
Difficulty: easy
Question: State different accounts who have account opening date before 1997 and own an amount of money greater than 3000USD
Generated SQL: SELECT DISTINCT a.account_id
FROM account AS a
JOIN trans AS t ON t.account_id = a.account_id
WHERE a.date < '1997-01-01'
GROUP BY a.account_id
HA

## Validate the reusable frontier baseline runner

In [15]:
from hashlib import sha256

from domain_adapt.frontier import (
    build_model_input,
    build_schema_context,
    load_baseline_config,
)


baseline_config_path = Path(
    "../configs/frontier_baseline_v1.json"
).resolve()

baseline_config = load_baseline_config(baseline_config_path)
module_schema_context = build_schema_context(conn)

# The reusable implementation must reproduce the already-measured prompt.
assert module_schema_context == schema_context
assert baseline_config["instructions"] == BASELINE_INSTRUCTIONS
assert baseline_config["max_output_tokens"] == 2000

expected_input = f"""
Database schema:

{schema_context}

Question:

{example["question"]}
""".strip()

assert (
    build_model_input(
        module_schema_context,
        example["question"],
    )
    == expected_input
)

config_sha256 = sha256(
    baseline_config_path.read_bytes()
).hexdigest()

schema_sha256 = sha256(
    module_schema_context.encode("utf-8")
).hexdigest()

print("Reusable frontier baseline validated.")
print("Config SHA-256:", config_sha256)
print("Schema SHA-256:", schema_sha256)
print("No model request made.")

Reusable frontier baseline validated.
Config SHA-256: 4331a4ae33d8b49a9b274f2ac030a4b737049a10565c7e3d6e92a84f69b1435c
Schema SHA-256: 612d6b1fc95531c1b62d63c850425f1d6d52dfecfdb7d8e919117d638bc43a4d
No model request made.


## Checkpoint 3: Blind frozen evaluation

The frontier baseline configuration is now fixed.

This evaluation makes exactly one model request for each frozen question. The
notebook does not display frozen questions, reference SQL, generated SQL,
result values, or per-question outcomes.

Detailed predictions are saved locally as a sealed run artifact. Only aggregate
metrics may be used when comparing systems. The frozen failures must not be
inspected or used to improve prompts, retrieval, glossaries, or training data.

In [16]:
from collections import Counter
from datetime import datetime, timezone
from hashlib import sha256
from statistics import median
from time import perf_counter

from domain_adapt.frontier import request_sql


EXPECTED_CONFIG_SHA256 = (
    "4331a4ae33d8b49a9b274f2ac030a4b737049a10565c7e3d6e92a84f69b1435c"
)
EXPECTED_SCHEMA_SHA256 = (
    "612d6b1fc95531c1b62d63c850425f1d6d52dfecfdb7d8e919117d638bc43a4d"
)

actual_config_sha256 = sha256(
    baseline_config_path.read_bytes()
).hexdigest()

actual_schema_sha256 = sha256(
    module_schema_context.encode("utf-8")
).hexdigest()

# These assertions run before any model request.
assert actual_config_sha256 == EXPECTED_CONFIG_SHA256
assert actual_schema_sha256 == EXPECTED_SCHEMA_SHA256
assert baseline_config["generation"]["attempts_per_question"] == 1
assert baseline_config["generation"]["candidate_sql_repair"] is False
assert baseline_config["generation"]["retry_incorrect_answers"] is False

frozen_ids = set(
    manifest["selection"]["frozen_eval"]["ids"]
)

frozen_examples = sorted(
    [
        row
        for row in records
        if row["db_name"] == "bird"
        and row["db_id"] == "financial"
        and row["question_id"] in frozen_ids
    ],
    key=lambda row: row["question_id"],
)

assert len(frozen_examples) == 24
assert {row["question_id"] for row in frozen_examples} == frozen_ids
assert frozen_ids.isdisjoint(working_ids)

sealed_run_path = Path(
    "../data/runs/sealed/"
    "gpt_5_2_minimal_frozen_eval_v1.json"
).resolve()


def save_sealed_run(run: dict) -> None:
    """Atomically save progress after every model request."""
    sealed_run_path.parent.mkdir(parents=True, exist_ok=True)

    temporary_path = sealed_run_path.with_suffix(".json.tmp")
    temporary_path.write_text(
        json.dumps(run, indent=2),
        encoding="utf-8",
    )
    temporary_path.replace(sealed_run_path)


if sealed_run_path.exists():
    sealed_run = json.loads(
        sealed_run_path.read_text(encoding="utf-8")
    )

    # Refuse to resume a run created with a different contract.
    assert (
        sealed_run["config_sha256"]
        == EXPECTED_CONFIG_SHA256
    )
    assert (
        sealed_run["schema_sha256"]
        == EXPECTED_SCHEMA_SHA256
    )
    assert sealed_run["split"] == "frozen_eval"

else:
    sealed_run = {
        "run_version": 1,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "benchmark": manifest["benchmark"],
        "split": "frozen_eval",
        "deployment_name": deployment_name,
        "declared_underlying_model": (
            baseline_config["declared_underlying_model"]
        ),
        "declared_model_version": (
            baseline_config["declared_model_version"]
        ),
        "config_sha256": EXPECTED_CONFIG_SHA256,
        "schema_sha256": EXPECTED_SCHEMA_SHA256,
        "attempts_per_question": 1,
        "candidate_sql_repair": False,
        "results": [],
    }
    save_sealed_run(sealed_run)


completed_ids = {
    result["question_id"]
    for result in sealed_run["results"]
}

pending_examples = [
    row
    for row in frozen_examples
    if row["question_id"] not in completed_ids
]

print("Already completed:", len(completed_ids))
print("Pending:", len(pending_examples))
print("Frozen contents will not be displayed.")


for frozen_example in pending_examples:
    started = perf_counter()

    try:
        response, generated_sql = request_sql(
            client=client,
            deployment_name=deployment_name,
            instructions=baseline_config["instructions"],
            schema_context=module_schema_context,
            question=frozen_example["question"],
            max_output_tokens=baseline_config["max_output_tokens"],
        )

        latency_seconds = perf_counter() - started

        compatible_reference_sql, applied_rules = (
            make_reference_compatible(frozen_example["SQL"])
        )

        comparison = compare_sql_results(
            conn=conn,
            reference_sql=compatible_reference_sql,
            candidate_sql=generated_sql,
        )

        usage = (
            response.usage.model_dump()
            if response.usage is not None
            else {}
        )

        result = {
            "question_id": frozen_example["question_id"],
            "difficulty": frozen_example["difficulty"],
            "generated_sql": generated_sql,
            "passed": comparison.equivalent,
            "reason": comparison.reason,
            "expected_row_count": comparison.expected_row_count,
            "candidate_row_count": comparison.candidate_row_count,
            "reference_compatibility_rules": applied_rules,
            "response_id": response.id,
            "response_status": response.status,
            "response_model": response.model,
            "latency_seconds": round(latency_seconds, 3),
            "usage": usage,
            "api_error": None,
        }

    except Exception as error:
        latency_seconds = perf_counter() - started

        # One attempt means an API failure is recorded, not retried.
        result = {
            "question_id": frozen_example["question_id"],
            "difficulty": frozen_example["difficulty"],
            "generated_sql": None,
            "passed": False,
            "reason": "model request failed",
            "expected_row_count": None,
            "candidate_row_count": None,
            "reference_compatibility_rules": None,
            "response_id": None,
            "response_status": None,
            "response_model": None,
            "latency_seconds": round(latency_seconds, 3),
            "usage": {},
            "api_error": f"{type(error).__name__}: {error}",
        }

    sealed_run["results"].append(result)
    save_sealed_run(sealed_run)

    # Do not reveal the ID or outcome.
    print(
        f"Completed {len(sealed_run['results'])}/"
        f"{len(frozen_examples)}"
    )

Already completed: 0
Pending: 24
Frozen contents will not be displayed.
Completed 1/24
Completed 2/24
Completed 3/24
Completed 4/24
Completed 5/24
Completed 6/24
Completed 7/24
Completed 8/24
Completed 9/24
Completed 10/24
Completed 11/24
Completed 12/24
Completed 13/24
Completed 14/24
Completed 15/24
Completed 16/24
Completed 17/24
Completed 18/24
Completed 19/24
Completed 20/24
Completed 21/24
Completed 22/24
Completed 23/24
Completed 24/24


In [17]:
frozen_results = sealed_run["results"]

assert len(frozen_results) == 24
assert {
    result["question_id"]
    for result in frozen_results
} == frozen_ids

passed_count = sum(
    result["passed"]
    for result in frozen_results
)

difficulty_summary = {}

for difficulty in ("easy", "medium", "hard"):
    subset = [
        result
        for result in frozen_results
        if result["difficulty"] == difficulty
    ]

    difficulty_summary[difficulty] = {
        "passed": sum(result["passed"] for result in subset),
        "total": len(subset),
        "accuracy": (
            sum(result["passed"] for result in subset)
            / len(subset)
        ),
    }

failure_reasons = Counter(
    result["reason"]
    for result in frozen_results
    if not result["passed"]
)

api_error_count = sum(
    result["api_error"] is not None
    for result in frozen_results
)

total_usage = {
    "input_tokens": sum(
        result["usage"].get("input_tokens", 0)
        for result in frozen_results
    ),
    "output_tokens": sum(
        result["usage"].get("output_tokens", 0)
        for result in frozen_results
    ),
    "reasoning_tokens": sum(
        result["usage"]
        .get("output_tokens_details", {})
        .get("reasoning_tokens", 0)
        for result in frozen_results
    ),
    "total_tokens": sum(
        result["usage"].get("total_tokens", 0)
        for result in frozen_results
    ),
}

median_latency = median(
    result["latency_seconds"]
    for result in frozen_results
)

sealed_run_sha256 = sha256(
    sealed_run_path.read_bytes()
).hexdigest()

print("Frozen evaluation complete.")
print(
    "Overall:",
    f"{passed_count}/{len(frozen_results)}",
    f"({passed_count / len(frozen_results):.1%})",
)
print("By difficulty:", difficulty_summary)
print("Failure reasons:", dict(failure_reasons))
print("API errors:", api_error_count)
print("Token usage:", total_usage)
print("Median latency seconds:", round(median_latency, 3))
print("Sealed-run SHA-256:", sealed_run_sha256)

Frozen evaluation complete.
Overall: 11/24 (45.8%)
By difficulty: {'easy': {'passed': 8, 'total': 12, 'accuracy': 0.6666666666666666}, 'medium': {'passed': 2, 'total': 9, 'accuracy': 0.2222222222222222}, 'hard': {'passed': 1, 'total': 3, 'accuracy': 0.3333333333333333}}
Failure reasons: {'unordered results differ': 7, 'ordered results differ': 3, 'column count differs: 1 != 5': 2, 'column count differs: 1 != 2': 1}
API errors: 0
Token usage: {'input_tokens': 7077, 'output_tokens': 2004, 'reasoning_tokens': 0, 'total_tokens': 9081}
Median latency seconds: 1.827
Sealed-run SHA-256: df06afc9484e92e842fdfebb0f4572247dc3dc65ee9dce4545ab3c414e30669e


Frontier model
+ one natural-language question
+ table/column/type schema only
+ one attempt
+ no examples, glossary, retrieval, or repair
= 45.8% execution accuracy